# Fake News Detection — Exploratory Data Analysis

Binary text classification of news articles as FAKE or REAL using the 'fake_or_real_news' dataset (6,335 articles). Headline and body text are combined into one field; TF-IDF features feed classic ML classifiers.

**Text column:** `content`  |  **Label column:** `label`

## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
from collections import Counter
sns.set_style('whitegrid')
pd.set_option('display.max_colwidth', 120)
%matplotlib inline

In [ ]:
from utils import load_data
df = load_data()
print('Shape:', df.shape)
df.head()

## 2. Dataset Overview

In [ ]:
print('Shape:', df.shape)
print('\nColumns:', list(df.columns))
print('\nDtypes:')
print(df.dtypes)

In [ ]:
print('Duplicate rows:', df.duplicated().sum())
print('Missing per column:'); print(df.isnull().sum())

## 3. Target / Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts = df['label'].value_counts()
counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Class counts'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)
axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Class proportions')
plt.tight_layout(); plt.show()
print('Class balance (%):')
print((counts / counts.sum() * 100).round(2))

## 4. Text Length Analysis

In [ ]:
df['char_count'] = df['content'].astype(str).str.len()
df['word_count'] = df['content'].astype(str).str.split().str.len()
df[['char_count', 'word_count']].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['char_count'].clip(upper=df['char_count'].quantile(0.99)).hist(bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Character count distribution'); axes[0].set_xlabel('chars')
df['word_count'].clip(upper=df['word_count'].quantile(0.99)).hist(bins=50, ax=axes[1], color='seagreen')
axes[1].set_title('Word count distribution'); axes[1].set_xlabel('words')
plt.tight_layout(); plt.show()

## 5. Text Length by Class

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
clip_val = df['word_count'].quantile(0.97)
sns.boxplot(x='label', y=df['word_count'].clip(upper=clip_val), data=df, ax=ax, palette='Set2')
ax.set_title('Word count by class'); ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()
df.groupby('label')['word_count'].describe().round(1)

## 6. Most Frequent Words (raw, pre-cleaning)

In [ ]:
import re
def quick_tokens(text):
    return re.findall(r'[a-zA-Z]+', str(text).lower())
all_words = Counter()
for t in df['content'].astype(str).sample(min(len(df), 5000), random_state=42):
    all_words.update(quick_tokens(t))
common = all_words.most_common(25)
plt.figure(figsize=(11, 5))
plt.barh([w for w, _ in common][::-1], [c for _, c in common][::-1], color='steelblue')
plt.title('25 most frequent raw words (sample of 5,000 docs)')
plt.tight_layout(); plt.show()

## 7. Word Clouds per Class

In [ ]:
from wordcloud import WordCloud
classes = df['label'].value_counts().index.tolist()[:4]
fig, axes = plt.subplots(1, len(classes), figsize=(5*len(classes), 4))
if len(classes) == 1: axes = [axes]
for ax, cls in zip(axes, classes):
    text = ' '.join(df[df['label'] == cls]['content'].astype(str).sample(min(2000, (df['label']==cls).sum()), random_state=42))
    wc = WordCloud(width=400, height=300, background_color='white', max_words=80).generate(text)
    ax.imshow(wc, interpolation='bilinear'); ax.axis('off'); ax.set_title(f'Class: {cls}')
plt.tight_layout(); plt.show()

## 8. Subject / Source Hints

In [ ]:
# Distinctive words: compare FAKE vs REAL vocabularies
from collections import Counter
import re
def toks(s): return re.findall(r'[a-z]+', str(s).lower())
fake_w, real_w = Counter(), Counter()
for t in df[df['label']=='FAKE']['content'].sample(min(1500,(df['label']=='FAKE').sum()), random_state=1):
    fake_w.update(toks(t))
for t in df[df['label']=='REAL']['content'].sample(min(1500,(df['label']=='REAL').sum()), random_state=1):
    real_w.update(toks(t))
print('Top FAKE-leaning words:', [w for w,_ in fake_w.most_common(15)])
print('Top REAL-leaning words:', [w for w,_ in real_w.most_common(15)])

## 9. Summary of Key Findings

In [ ]:
summary = pd.DataFrame({
    'Metric': ['Total documents', 'Number of classes', 'Most common class',
               'Least common class', 'Mean word count', 'Median word count',
               'Max word count', 'Duplicate rows'],
    'Value': [
        len(df), df['label'].nunique(),
        str(df['label'].value_counts().idxmax()),
        str(df['label'].value_counts().idxmin()),
        round(df['word_count'].mean(), 1),
        int(df['word_count'].median()),
        int(df['word_count'].max()),
        int(df.duplicated().sum()),
    ],
})
summary